https://tianchi.aliyun.com/notebook/144452

## Task3 多路召回

所谓的“多路召回”策略，就是指采用不同的策略、特征或简单模型，分别召回一部分候选集，然后把候选集混合在一起供后续排序模型使用，可以明显的看出，“多路召回策略”是在“计算速度”和“召回率”之间进行权衡的结果。其中，各种简单策略保证候选集的快速召回，从不同角度设计的策略保证召回率接近理想的状态，不至于损伤排序效果。在多路召回中，每个策略之间毫不相关，所以一般可以写并发多线程同时进行，这样可以更加高效。

可以使用多种不同的策略来获取用户排序的候选商品集合，而具体使用哪些召回策略其实是与业务强相关的 ，针对不同的任务就会有对于该业务真实场景下需要考虑的召回规则。例如新闻推荐，召回规则可以是“热门新闻”、“作者召回”、“关键词召回”、“主题召回“、”协同过滤召回“等等。

## 导包

In [1]:
# ============================================================
# 1. Python 标准库与环境配置
# ============================================================
import os
import sys
import math
import pickle
import random
import inspect
import warnings
import collections

from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

# 必须在导入 TensorFlow / DeepCTR / DeepMatch 前设置
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# 将项目父目录加入模块搜索路径，支持 metric 导入
sys.path.append(str(Path.cwd().parent))


# ============================================================
# 2. 数据处理与向量检索
# ============================================================
import pandas as pd
import numpy as np
import faiss

from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler, LabelEncoder


# ============================================================
# 3. TensorFlow / DeepCTR / DeepMatch
# ============================================================
import tensorflow as tf

from tf_keras import backend as K
from tf_keras.models import Model
from tf_keras.utils import pad_sequences

from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from deepmatch.models import YoutubeDNN
from deepmatch.utils import sampledsoftmaxloss


# ============================================================
# 4. 离线评估模块
# ============================================================
from metric.data import prepare_evaluation
from metric.evaluation import evaluate_predictions, save_predictions



warnings.filterwarnings("ignore")



In [2]:
data_path = '../data/'
save_path = './results/'
eval_cache_path = '../cache/offline_eval'
# 做召回评估的一个标志, 如果不进行评估就是直接使用全量数据进行召回
metric_recall = True

## 数据读取

In [3]:
def get_all_click_sample(data_path, sample_nums=10000):
    """
    从训练数据中抽样一部分数据调试
    """
    all_click = pd.read_csv(data_path + 'train_click_log.csv')

    all_user_ids = all_click.user_id.unique()
    sample_user_ids = np.random.choice(
        all_user_ids,
        size=sample_nums,
        replace=False
    )

    all_click = all_click[
        all_click['user_id'].isin(sample_user_ids)
    ]

    all_click = all_click.drop_duplicates(
        ['user_id', 'click_article_id', 'click_timestamp']
    )

    return all_click

# 读取文章基本属性
def get_item_info_df(data_path):
    item_info_df = pd.read_csv(
        data_path + 'articles.csv'
    )

    # 与点击日志中的 click_article_id 保持一致
    item_info_df = item_info_df.rename(
        columns={'article_id': 'click_article_id'}
    )

    return item_info_df


# 读取文章 Embedding 数据
def get_item_emb_dict(data_path):
    item_emb_df = pd.read_csv(
        data_path + 'articles_emb.csv'
    )

    item_emb_cols = [
        x for x in item_emb_df.columns
        if 'emb' in x
    ]

    item_emb_np = np.ascontiguousarray(
        item_emb_df[item_emb_cols]
    )

    # 处理：单位化
    item_emb_np = (
        item_emb_np
        / np.linalg.norm(
            item_emb_np,
            axis=1,
            keepdims=True
        )
    )

    item_emb_dict = dict(
        zip(
            item_emb_df['article_id'],
            item_emb_np
        )
    )

    pickle.dump(
        item_emb_dict,
        open(
            save_path + 'item_content_emb.pkl',
            'wb'
        )
    )

    return item_emb_dict


# 时间戳归一化函数
max_min_scaler = lambda x: (
    (x - np.min(x))
    / (np.max(x) - np.min(x))
)


In [4]:
train_clicks = pd.read_csv(
    data_path + 'train_click_log.csv'
)

fit_clicks, query_clicks, targets = prepare_evaluation(
    train_clicks,
)

# copy 是必要的，因为下面会修改 click_timestamp
all_click_df = fit_clicks.copy()

# 对时间戳归一化，用于后续关联规则权重
all_click_df['click_timestamp'] = (
    all_click_df[['click_timestamp']]
    .apply(max_min_scaler)
)


item_info_df = get_item_info_df(data_path)

item_emb_dict = get_item_emb_dict(data_path)

## 工具函数

### 获取用户-文章-时间函数

这个在基于关联规则的用户协同过滤的时候会用到

In [5]:
# 根据时间获取文章被点击的用户序列
# {item1: [(user1, time1), (user2, time2), ...], ...}
def get_item_user_time_dict(click_df):
    def make_user_time_pair(df):
        return list(
            zip(
                df['user_id'],
                df['click_timestamp']
            )
        )

    click_df = click_df.sort_values('click_timestamp')

    item_user_time_df = (
        click_df
        .groupby('click_article_id')[['user_id', 'click_timestamp']]
        .apply(lambda x: make_user_time_pair(x))
        .reset_index()
        .rename(columns={0: 'user_time_list'})
    )

    item_user_time_dict = dict(
        zip(
            item_user_time_df['click_article_id'],
            item_user_time_df['user_time_list']
        )
    )

    return item_user_time_dict

### 获取文章-用户-时间函数

这个在基于关联规则的用户协同过滤的时候会用到

In [6]:
# 1. 用户 -> [(文章, 点击时间), ...]
def get_user_item_time(click_df):
    click_df = click_df.sort_values('click_timestamp')

    def make_item_time_pair(df):
        return list(zip(
            df['click_article_id'],
            df['click_timestamp']
        ))

    user_item_time_df = (
        click_df
        .groupby('user_id')[['click_article_id', 'click_timestamp']]
        .apply(lambda x: make_item_time_pair(x))
        .reset_index()
        .rename(columns={0: 'item_time_list'})
    )

    user_item_time_dict = dict(zip(
        user_item_time_df['user_id'],
        user_item_time_df['item_time_list']
    ))

    return user_item_time_dict

In [7]:
# 2. 文章 -> [(用户, 点击时间), ...]
def get_item_user_time_dict(click_df):
    click_df = click_df.sort_values('click_timestamp')

    def make_user_time_pair(df):
        return list(zip(
            df['user_id'],
            df['click_timestamp']
        ))

    item_user_time_df = (
        click_df
        .groupby('click_article_id')[['user_id', 'click_timestamp']]
        .apply(lambda x: make_user_time_pair(x))
        .reset_index()
        .rename(columns={0: 'user_time_list'})
    )

    item_user_time_dict = dict(zip(
        item_user_time_df['click_article_id'],
        item_user_time_df['user_time_list']
    ))

    return item_user_time_dict

### 获取历史和最后一次点击（离线评估未使用）

在评估召回结果， 特征工程和制作标签转成监督学习测试集的时候可能会用到，仅参考写法

In [8]:
def get_hist_and_last_click(all_click):
    all_click = all_click.sort_values(
        by=['user_id', 'click_timestamp']
    )

    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = (
        all_click
        .groupby('user_id')
        .apply(hist_func)
        .reset_index(drop=True)
    )

    return click_hist_df, click_last_df

### 获取文章属性特征

In [9]:
def get_item_info_dict(item_info_df):
    max_min_scaler = lambda x: (
        (x - np.min(x)) /
        (np.max(x) - np.min(x))
    )

    item_info_df['created_at_ts'] = (
        item_info_df[['created_at_ts']]
        .apply(max_min_scaler)
    )

    item_type_dict = dict(
        zip(
            item_info_df['click_article_id'],
            item_info_df['category_id']
        )
    )

    item_words_dict = dict(
        zip(
            item_info_df['click_article_id'],
            item_info_df['words_count']
        )
    )

    item_created_time_dict = dict(
        zip(
            item_info_df['click_article_id'],
            item_info_df['created_at_ts']
        )
    )

    return (
        item_type_dict,
        item_words_dict,
        item_created_time_dict
    )

### 获取用户历史点击的文章信息

- 注意fit_clicks只有三列(user_id、click_article_id、click_timestamp)，还需要category_id、words_count、created_at_ts

- 调用之前必须：```all_click_df_ = all_click_df.merge(
    item_info_df,
    how='left',
    on='click_article_id'
)```   
然后```(
    user_hist_item_typs_dict,
    user_hist_item_ids_dict,
    user_hist_item_words_dict,
    user_last_item_created_time_dict
) = get_user_hist_item_info_dict(all_click_df_)```


In [10]:
def get_user_hist_item_info_dict(all_click):

    # 用户历史点击文章类型集合
    user_hist_item_typs = (
        all_click
        .groupby('user_id')['category_id']
        .agg(set)
        .reset_index()
    )

    user_hist_item_typs_dict = dict(
        zip(
            user_hist_item_typs['user_id'],
            user_hist_item_typs['category_id']
        )
    )

    # 用户历史点击文章集合
    user_hist_item_ids_dict = (
        all_click
        .groupby('user_id')['click_article_id']
        .agg(set)
        .reset_index()
    )

    user_hist_item_ids_dict = dict(
        zip(
            user_hist_item_ids_dict['user_id'],
            user_hist_item_ids_dict['click_article_id']
        )
    )

    # 用户历史文章平均字数
    user_hist_item_words = (
        all_click
        .groupby('user_id')['words_count']
        .agg('mean')
        .reset_index()
    )

    user_hist_item_words_dict = dict(
        zip(
            user_hist_item_words['user_id'],
            user_hist_item_words['words_count']
        )
    )

    # 用户最后一次点击文章的创建时间
    all_click_ = all_click.sort_values('click_timestamp')

    user_last_item_created_time = (
        all_click_
        .groupby('user_id')['created_at_ts']
        .apply(lambda x: x.iloc[-1])
        .reset_index()
    )

    max_min_scaler = lambda x: (
        (x - np.min(x)) /
        (np.max(x) - np.min(x))
    )

    user_last_item_created_time['created_at_ts'] = (
        user_last_item_created_time[['created_at_ts']]
        .apply(max_min_scaler)
    )

    user_last_item_created_time_dict = dict(
        zip(
            user_last_item_created_time['user_id'],
            user_last_item_created_time['created_at_ts']
        )
    )

    return (
        user_hist_item_typs_dict,
        user_hist_item_ids_dict,
        user_hist_item_words_dict,
        user_last_item_created_time_dict
    )

### 获取点击次数最多的topk个文章

In [11]:
def get_item_topk_click(click_df, k):
    topk_click = (
        click_df['click_article_id']
        .value_counts()
        .index[:k]
    )

    return topk_click

## 定义多路召回字典

In [12]:
# 获取文章的属性信息，保存成字典的形式方便查询
item_type_dict, item_words_dict, item_created_time_dict = \
    get_item_info_dict(item_info_df)


In [13]:
# 定义一个多路召回的字典，将各路召回的结果都保存在这个字典当中
user_multi_recall_dict = {
    'itemcf_sim_itemcf_recall': {},
    'embedding_sim_item_recall': {},
    'youtubednn_recall': {},
    'youtubednn_usercf_recall': {},
    'cold_start_recall': {}
}

In [14]:
# 提取最后一次点击作为召回评估，如果不需要做召回评估直接使用全量的训练集进行召回(线下验证模型)

trn_hist_click_df = all_click_df
trn_last_click_df = targets

# 后续真正产生离线推荐时只预测 query_clicks 中的用户
query_users = query_clicks['user_id'].unique()

## 召回效果评估函数（未使用）

提供的召回评估方法，主要不同在于hit rate计算分母为成功产生推荐的用户数；当前公共评估采用targets的固定用户数。

In [15]:
# 依次评估召回的前10, 20, 30, 40, 50个文章中的击中率
def metrics_recall(user_recall_items_dict, trn_last_click_df, topk=5):
    last_click_item_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))
    user_num = len(user_recall_items_dict)
    
    for k in range(10, topk+1, 10):
        hit_num = 0
        for user, item_list in user_recall_items_dict.items():
            # 获取前k个召回的结果
            tmp_recall_items = [x[0] for x in user_recall_items_dict[user][:k]]
            if last_click_item_dict[user] in set(tmp_recall_items):
                hit_num += 1
        
        hit_rate = round(hit_num * 1.0 / user_num, 5)
        print(' topk: ', k, ' : ', 'hit_num: ', hit_num, 'hit_rate: ', hit_rate, 'user_num : ', user_num)

## 相似性矩阵

协同过滤与向量检索，分为user2user和item2item

### itemcf i2i_sim

KDD2020的去偏商品推荐，使用关联规则，考虑到了：

- 用户点击的时间权重

- 用户点击的顺序权重

- 文章创建的时间权重

In [16]:
def itemcf_sim(df, item_created_time_dict):
    """
        文章与文章之间的相似性矩阵计算
        :param df: 数据表
        :item_created_time_dict:  文章创建时间的字典
        return : 文章与文章的相似性矩阵
        
        思路: 基于物品的协同过滤(详细请参考上一期推荐系统基础的组队学习) + 关联规则
    """
    
    user_item_time_dict = get_user_item_time(df)
    
    # 计算物品相似度
    i2i_sim = {}
    item_cnt = defaultdict(int)
    for user, item_time_list in tqdm(user_item_time_dict.items()):
        # 在基于商品的协同过滤优化的时候可以考虑时间因素
        for loc1, (i, i_click_time) in enumerate(item_time_list):
            item_cnt[i] += 1
            i2i_sim.setdefault(i, {})
            for loc2, (j, j_click_time) in enumerate(item_time_list):
                if(i == j):
                    continue
                    
                # 考虑文章的正向顺序点击和反向顺序点击    
                loc_alpha = 1.0 if loc2 > loc1 else 0.7
                # 位置信息权重，其中的参数可以调节
                loc_weight = loc_alpha * (0.9 ** (np.abs(loc2 - loc1) - 1))
                # 点击时间权重，其中的参数可以调节
                click_time_weight = np.exp(0.7 ** np.abs(i_click_time - j_click_time))
                # 两篇文章创建时间的权重，其中的参数可以调节
                created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                i2i_sim[i].setdefault(j, 0)
                # 考虑多种因素的权重计算最终的文章之间的相似度
                i2i_sim[i][j] += loc_weight * click_time_weight * created_time_weight / math.log(len(item_time_list) + 1)
                
    i2i_sim_ = i2i_sim.copy()
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij / math.sqrt(item_cnt[i] * item_cnt[j])
    
    # 将得到的相似性矩阵保存到本地
    pickle.dump(i2i_sim_, open(save_path + 'itemcf_i2i_sim.pkl', 'wb'))
    
    return i2i_sim_

In [17]:
i2i_sim = itemcf_sim(
    all_click_df,
    item_created_time_dict
)

100%|██████████| 200000/200000 [00:42<00:00, 4708.99it/s]


In [18]:
print('train users:', train_clicks['user_id'].nunique())
print('fit users:', fit_clicks['user_id'].nunique())
print('query users:', query_clicks['user_id'].nunique())
print('target users:', targets['user_id'].nunique())

print('train clicks:', len(train_clicks))
print('fit clicks:', len(fit_clicks))
print('removed clicks:', len(train_clicks) - len(fit_clicks))

train users: 200000
fit users: 200000
query users: 50000
target users: 50000
train clicks: 1112623
fit clicks: 1062623
removed clicks: 50000


### usercf u2u_sim

计算用户相似度指标，使用简单的关联规则，比如用户活跃度权重，这里将用户点击次数作为活跃度指标

In [19]:
def get_user_activate_degree_dict(all_click_df):
    all_click_df_ = (
        all_click_df
        .groupby('user_id')['click_article_id']
        .count()
        .reset_index()
    )

    # 用户活跃度归一化
    mm = MinMaxScaler()

    all_click_df_['click_article_id'] = (
        mm.fit_transform(
            all_click_df_[['click_article_id']]
        )
    )

    user_activate_degree_dict = dict(
        zip(
            all_click_df_['user_id'],
            all_click_df_['click_article_id']
        )
    )

    return user_activate_degree_dict

In [20]:
def usercf_sim(
    all_click_df,
    user_activate_degree_dict,
    sample_user_num=None
):
    """
        用户相似性矩阵计算
        :param all_click_df: 数据表
        :param user_activate_degree_dict: 用户活跃度字典
        :param sample_user_num: 采样用户数，None 表示不采样
        return: 用户相似性矩阵
    """

    # 采样用户，保留被采样用户的全部点击记录
    if sample_user_num is not None:
        all_user_ids = all_click_df['user_id'].unique()

        if sample_user_num < len(all_user_ids):
            sample_user_ids = np.random.RandomState(42).choice(
                all_user_ids,
                size=sample_user_num,
                replace=False
            )

            all_click_df = all_click_df[
                all_click_df['user_id'].isin(sample_user_ids)
            ]

    # 获取 item -> [(user, time), ...]
    item_user_time_dict = get_item_user_time_dict(all_click_df)

    u2u_sim = {}
    user_cnt = defaultdict(int)

    for item, user_time_list in tqdm(
        item_user_time_dict.items()
    ):
        for u, click_time in user_time_list:
            user_cnt[u] += 1
            u2u_sim.setdefault(u, {})

            for v, click_time in user_time_list:
                if u == v:
                    continue

                activate_weight = (
                    100 * 0.5 *
                    (
                        user_activate_degree_dict[u] +
                        user_activate_degree_dict[v]
                    )
                )

                # 避免原代码可能出现的 KeyError
                u2u_sim[u][v] = (
                    u2u_sim[u].get(v, 0)
                    + activate_weight
                    / math.log(len(user_time_list) + 1)
                )

    u2u_sim_ = u2u_sim.copy()

    for u, related_users in u2u_sim.items():
        for v, wij in related_users.items():
            u2u_sim_[u][v] = (
                wij /
                math.sqrt(
                    user_cnt[u] *
                    user_cnt[v]
                )
            )

    pickle.dump(
        u2u_sim_,
        open(
            save_path + 'usercf_u2u_sim.pkl',
            'wb'
        )
    )

    return u2u_sim_

In [21]:
user_activate_degree_dict = get_user_activate_degree_dict(all_click_df)
u2u_sim = usercf_sim(all_click_df, user_activate_degree_dict, sample_user_num = 50000)

100%|██████████| 13257/13257 [00:52<00:00, 253.96it/s]


### item embedding sim

使用Embedding计算item相似度，为了后续冷启动时可以获取未出现在点击数据中的文章，使用faiss

- faiss是Facebook的AI团队开源的一套用于做聚类或者相似性搜索的软件库，底层是用C++实现。Faiss因为超级优越的性能，被广泛应用于推荐相关的业务当中.

- faiss工具包一般使用在推荐系统中的向量召回部分。在做向量召回的时候要么是u2u,u2i或者i2i，这里的u和i指的是user和item.我们知道在实际的场景中user和item的数量都是海量的，我们最容易想到的基于向量相似度的召回就是使用两层循环遍历user列表或者item列表计算两个向量的相似度，但是这样做在面对海量数据是不切实际的，faiss就是用来加速计算某个查询向量最相似的topk个索引向量

- faiss查询原理：使用PCA和PQ(Product quantization乘积量化)两种技术进行向量压缩和编码，当然还使用了其他的技术进行优化，但是PCA和PQ是其中最核心部分。


In [22]:
# 向量检索相似度计算
# topk指的是每个item, faiss搜索后返回最相似的topk个item
def embdding_sim(click_df, item_emb_df, save_path, topk):
    """
        基于内容的文章embedding相似性矩阵计算
        :param click_df: 数据表
        :param item_emb_df: 文章的embedding
        :param save_path: 保存路径
        :patam topk: 找最相似的topk篇
        return 文章相似性矩阵
        
        思路: 对于每一篇文章， 基于embedding的相似性返回topk个与其最相似的文章， 只不过由于文章数量太多，这里用了faiss进行加速
    """
    
    # 文章索引与文章id的字典映射
    item_idx_2_rawid_dict = dict(zip(item_emb_df.index, item_emb_df['article_id']))
    
    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols].values, dtype=np.float32)
    # 向量进行单位化
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)
    
    # 建立faiss索引
    item_index = faiss.IndexFlatIP(item_emb_np.shape[1])
    item_index.add(item_emb_np)
    # 相似度查询，给每个索引位置上的向量返回topk个item以及相似度
    sim, idx = item_index.search(item_emb_np, topk) # 返回的是列表
    
    # 将向量检索的结果保存成原始id的对应关系
    item_sim_dict = collections.defaultdict(dict)
    for target_idx, sim_value_list, rele_idx_list in tqdm(zip(range(len(item_emb_np)), sim, idx)):
        target_raw_id = item_idx_2_rawid_dict[target_idx]
        # 从1开始是为了去掉商品本身, 所以最终获得的相似商品只有topk-1
        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]): 
            rele_raw_id = item_idx_2_rawid_dict[rele_idx]
            item_sim_dict[target_raw_id][rele_raw_id] = item_sim_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + sim_value
    
    # 保存i2i相似度矩阵
    pickle.dump(item_sim_dict, open(save_path + 'emb_i2i_sim.pkl', 'wb'))   
    
    return item_sim_dict

In [23]:
item_emb_df = pd.read_csv(data_path + '/articles_emb.csv')
emb_i2i_sim = embdding_sim(all_click_df, item_emb_df, save_path, topk=10) # topk可以自行设置

364047it [00:07, 46976.21it/s]


## 召回

面对36万篇文章， 20多万用户的推荐， 我们又有哪些策略来缩减问题的规模？ 我们就可以再召回阶段筛选出用户对于点击文章的候选集合， 从而降低问题的规模。召回常用的策略：

- Youtube DNN召回

- 基于文章的召回
  - 文章的协同过滤
  - 基于文章embedding召回
  
- 基于用户的召回
  - 用户的协同过滤
  - 用户embedding

上面的各种召回方式一部分在基于用户已经看得文章的基础上去召回与这些文章相似的一些文章， 而这个相似性的计算方式不同， 就得到了不同的召回方式， 比如文章的协同过滤， 文章内容的embedding等。还有一部分是根据用户的相似性进行推荐，对于某用户推荐与其相似的其他用户看过的文章，比如用户的协同过滤和用户embedding。 还有一种思路是类似矩阵分解的思路，先计算出用户和文章的embedding之后，就可以直接算用户和文章的相似度， 根据这个相似度进行推荐， 比如YouTube DNN。

### YoutubeDNN召回

直接获取用户召回的候选文章列表

In [24]:
def gen_data_set(data, negsample=0):
    """
    沿用原文按用户点击顺序构造滑窗监督样本。

    1. fit_clicks 已经留出公共 targets，不再留内部 test_set。
    2. 单点击用户不构造 item -> item 的自预测样本。
    3. 模型使用 sampled softmax 时，调用 negsample=0。

    返回：
        train_set: [(user_id, hist, target_item, label, hist_len), ...]
        test_set: 保留原函数的第二返回值接口，这里为空列表
    """
    data = data.sort_values(
        ["user_id", "click_timestamp"],
        kind="mergesort"
    )

    item_ids = data["click_article_id"].unique()

    train_set = []
    test_set = []

    for reviewerID, hist in tqdm(
        data.groupby("user_id", sort=False),
        desc="Build YoutubeDNN samples"
    ):
        pos_list = hist["click_article_id"].tolist()

        if negsample > 0:
            candidate_set = list(set(item_ids) - set(pos_list))
            neg_list = np.random.choice(
                candidate_set,
                size=len(pos_list) * negsample,
                replace=True
            )

        # 单点击用户缺少“历史 -> 下一次点击”监督关系
        # 仍保留在 data 中，之后照样可以提取 user embedding
        if len(pos_list) == 1:
            continue

        for i in range(1, len(pos_list)):
            hist_items = pos_list[:i][::-1]

            train_set.append((
                reviewerID,
                hist_items,
                pos_list[i],
                1,
                len(hist_items)
            ))

            for negi in range(negsample):
                train_set.append((
                    reviewerID,
                    hist_items,
                    neg_list[i * negsample + negi],
                    0,
                    len(hist_items)
                ))

    random.shuffle(train_set)
    random.shuffle(test_set)

    return train_set, test_set


def gen_model_input(train_set, user_profile, seq_max_len):
    """

    hist_len 必须与实际截断后的序列长度一致。
    """
    train_uid = np.array(
        [line[0] for line in train_set],
        dtype=np.int32
    )

    train_seq = [
        line[1] for line in train_set
    ]

    train_iid = np.array(
        [line[2] for line in train_set],
        dtype=np.int32
    )

    train_label = np.array(
        [line[3] for line in train_set],
        dtype=np.float32
    )

    train_hist_len = np.minimum(
        np.array([line[4] for line in train_set]),
        seq_max_len
    ).astype(np.int32)

    train_seq_pad = pad_sequences(
        train_seq,
        maxlen=seq_max_len,
        padding="post",
        truncating="post",
        value=0
    ).astype(np.int32)

    train_model_input = {
        "user_id": train_uid,
        "click_article_id": train_iid,
        "hist_article_id": train_seq_pad,
        "hist_len": train_hist_len
    }

    return train_model_input, train_label

In [27]:
def youtubednn_u2i_dict(data, query_clicks, topk=50):
    """
    YouTubeDNN U2I 召回。

    data:
        完整 fit_clicks（可以是时间戳归一化后的 all_click_df）。
        仅使用其中的可见点击记录训练模型。

    query_clicks:
        需要进行离线召回的用户及其可见点击历史。
        只用于确定用户范围和过滤已点击文章。

    topk:
        每个 query user 最多返回多少篇未点击文章。

    保存：
        user_youtube_emb.pkl
        item_youtube_emb.pkl
        youtube_u2i_dict.pkl
    """

    # 不能原地覆盖 all_click_df
    data = data.copy()

    SEQ_LEN = 30
    embedding_dim = 16

    # 原文先记录原始 ID，再进行 LabelEncoder
    user_profile_ = data[["user_id"]].drop_duplicates("user_id")
    item_profile_ = data[
        ["click_article_id"]
    ].drop_duplicates("click_article_id")

    # ----------------------------
    # 1. 类别编码
    # ----------------------------
    features = ["click_article_id", "user_id"]
    feature_max_idx = {}
    index_2_rawid = {}

    for feature in features:
        lbe = LabelEncoder()

        # item 使用 +1，保留 ID=0 给 padding
        # user 保留原文从 0 开始的编码习惯
        offset = 1 if feature == "click_article_id" else 0

        data[feature] = (
            lbe.fit_transform(data[feature]) + offset
        )  # type: ignore

        feature_max_idx[feature] = (
            int(data[feature].max()) + 1
        )

        index_2_rawid[feature] = {
            int(encoded_id + offset): int(raw_id)
            for encoded_id, raw_id in enumerate(lbe.classes_)
        }

    user_index_2_rawid = index_2_rawid["user_id"]
    item_index_2_rawid = index_2_rawid["click_article_id"]

    user_profile = data[
        ["user_id"]
    ].drop_duplicates("user_id")

    item_profile = data[
        ["click_article_id"]
    ].drop_duplicates("click_article_id")

    # 原文中的 raw profile 仍可保留用于核对
    assert len(user_profile) == len(user_profile_)
    assert len(item_profile) == len(item_profile_)

    # 确保用户历史有确定的时间顺序
    data = data.sort_values(
        ["user_id", "click_timestamp"],
        kind="mergesort"
    )

    # ----------------------------
    # 2. 生成滑窗训练样本
    # ----------------------------
    train_set, test_set = gen_data_set(data, negsample=0)

    if not train_set:
        raise ValueError(
            "没有有效的 YouTubeDNN 滑窗训练样本"
        )

    train_model_input, train_label = gen_model_input(
        train_set, user_profile, SEQ_LEN
    )

    # ----------------------------
    # 3. 构造全部用户的完整可见历史
    # ----------------------------
    # 这里不使用原文内部 test_set。
    # 直接用 fit_clicks 的全部可见历史提取 user embedding，
    # 保留 20 万用户的 embedding，供后续 U2U 使用。
    infer_set = []

    for uid, hist_df in data.groupby("user_id", sort=False):
        hist_items = hist_df["click_article_id"].tolist()

        infer_set.append((
            uid,
            hist_items[::-1],
            hist_items[-1],   # 仅作 gen_model_input 的格式占位
            1,
            len(hist_items)
        ))

    infer_model_input, _ = gen_model_input(
        infer_set, user_profile, SEQ_LEN
    )

    # ----------------------------
    # 4. 定义 DeepMatch 特征
    # ----------------------------
    user_feature_columns = [
        SparseFeat(
            "user_id",
            feature_max_idx["user_id"],
            embedding_dim
        ),
        VarLenSparseFeat(
            SparseFeat(
                "hist_article_id",
                feature_max_idx["click_article_id"],
                embedding_dim,
                embedding_name="click_article_id"
            ),
            SEQ_LEN,
            "mean",
            "hist_len"
        )
    ]

    item_feature_columns = [
        SparseFeat(
            "click_article_id",
            feature_max_idx["click_article_id"],
            embedding_dim
        )
    ]

        # ----------------------------
    # 5. 定义 YoutubeDNN 模型
    # ----------------------------

    # 修复 DeepMatch get_item_embedding 的 TypeSpec 兼容问题
    import importlib
    import tensorflow as tf
    from tf_keras.layers import Lambda

    def get_item_embedding_compat(
        item_embedding,
        item_input_layer
    ):
        # 将 tf.gather 移出 Lambda
        item_out = tf.gather(
            item_embedding,
            item_input_layer
        )

        return Lambda(
            lambda x: tf.squeeze(x, axis=1)
        )(item_out)

    # 修改 YoutubeDNN 模块内部使用的函数引用
    youtube_module = importlib.import_module(
        "deepmatch.models.youtubednn"
    )

    youtube_module.get_item_embedding = (
        get_item_embedding_compat
    )

    assert YoutubeDNN is youtube_module.YoutubeDNN

    # 兼容不同 DeepMatch 版本
    model_signature = inspect.signature(YoutubeDNN)

    if "sampler_config" in model_signature.parameters:
        from deepmatch.utils import NegativeSampler

        train_counter = Counter(
            train_model_input["click_article_id"].tolist()
        )

        item_count = [
            train_counter.get(i, 0)
            for i in range(
                feature_max_idx["click_article_id"]
            )
        ]

        sampler_config = NegativeSampler(
            "frequency",
            num_sampled=5,
            item_name="click_article_id",
            item_count=item_count
        )

        model = YoutubeDNN(
            user_feature_columns,
            item_feature_columns,
            sampler_config=sampler_config,
            user_dnn_hidden_units=(64, embedding_dim)
        )

    elif "num_sampled" in model_signature.parameters:
        model = YoutubeDNN(
            user_feature_columns,
            item_feature_columns,
            num_sampled=5,
            user_dnn_hidden_units=(64, embedding_dim)
        )

    else:
        raise RuntimeError(
            "当前 YoutubeDNN 接口无法识别："
            f"{model_signature}"
        )

    model.compile(
        optimizer="adam",
        loss=sampledsoftmaxloss
    )

    # ----------------------------
    # 6. 模型训练
    # ----------------------------
    history = model.fit(
        train_model_input,
        train_label,
        batch_size=256,
        epochs=1,
        verbose=1,
        validation_split=0.0
    )

    # ----------------------------
    # 7. 提取 User / Item Embedding
    # ----------------------------
    all_item_model_input = {
        "click_article_id":
            item_profile["click_article_id"].to_numpy(
                dtype=np.int32
            )
    }

    user_embedding_model = Model(
        inputs=model.user_input,
        outputs=model.user_embedding
    )

    item_embedding_model = Model(
        inputs=model.item_input,
        outputs=model.item_embedding
    )

    user_embs = user_embedding_model.predict(
        infer_model_input,
        batch_size=2 ** 12,
        verbose=1
    )

    item_embs = item_embedding_model.predict(
        all_item_model_input,
        batch_size=2 ** 12,
        verbose=1
    )

    # 与原文一致：归一化后做内积搜索
    user_embs = np.ascontiguousarray(
        user_embs, dtype=np.float32
    )

    item_embs = np.ascontiguousarray(
        item_embs, dtype=np.float32
    )

    user_embs /= np.maximum(
        np.linalg.norm(
            user_embs, axis=1, keepdims=True
        ),
        1e-12
    )

    item_embs /= np.maximum(
        np.linalg.norm(
            item_embs, axis=1, keepdims=True
        ),
        1e-12
    )

    # ----------------------------
    # 8. Embedding 恢复原始 ID 并保存
    # ----------------------------
    # 必须按 predict() 输入行顺序来映射。
    raw_user_ids = np.array([
        user_index_2_rawid[int(uid)]
        for uid in infer_model_input["user_id"]
    ], dtype=np.int64)

    raw_item_ids = np.array([
        item_index_2_rawid[int(item_id)]
        for item_id in all_item_model_input[
            "click_article_id"
        ]
    ], dtype=np.int64)

    raw_user_id_emb_dict = {
        int(uid): emb
        for uid, emb in zip(raw_user_ids, user_embs)
    }

    raw_item_id_emb_dict = {
        int(item_id): emb
        for item_id, emb in zip(raw_item_ids, item_embs)
    }

    pickle.dump(
        raw_user_id_emb_dict,
        open(save_path + "user_youtube_emb.pkl", "wb")
    )

    pickle.dump(
        raw_item_id_emb_dict,
        open(save_path + "item_youtube_emb.pkl", "wb")
    )

    print("User embedding:", user_embs.shape)
    print("Item embedding:", item_embs.shape)

    # ----------------------------
    # 9. Faiss U2I 检索
    # ----------------------------
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(item_embs)

    query_user_ids = query_clicks[
        "user_id"
    ].unique()

    query_mask = np.isin(
        raw_user_ids,
        query_user_ids
    )

    query_embs = np.ascontiguousarray(
        user_embs[query_mask],
        dtype=np.float32
    )

    query_raw_user_ids = raw_user_ids[query_mask]

    if len(query_raw_user_ids) != len(query_user_ids):
        raise ValueError(
            "query_clicks 中存在未能映射到模型的用户"
        )

    # 用用户完整可见历史过滤已点击文章
    user_hist_items = (
        query_clicks.groupby("user_id")[
            "click_article_id"
        ]
        .agg(set)
        .to_dict()
    )

    # 比 topk 多搜索一些，用于过滤已读文章
    max_hist_len = (
        query_clicks.groupby("user_id").size().max()
    )

    search_k = min(
        len(item_embs),
        topk + int(max_hist_len)
    )

    sim, idx = index.search(
        query_embs,
        search_k
    )

    # ----------------------------
    # 10. 恢复原始文章 ID
    # ----------------------------
    user_recall_items_dict = {}

    for raw_uid, sim_values, item_indices in tqdm(
        zip(query_raw_user_ids, sim, idx),
        total=len(query_raw_user_ids),
        desc="YoutubeDNN U2I recall"
    ):
        raw_uid = int(raw_uid)
        seen_items = user_hist_items.get(raw_uid, set())

        recall_items = []

        # U2I 没有“自己”这个 item。
        # 因此不能像原文一样跳过第一个结果。
        for item_idx, score in zip(
            item_indices, sim_values
        ):
            if item_idx < 0:
                continue

            raw_item_id = int(
                raw_item_ids[int(item_idx)]
            )

            if raw_item_id in seen_items:
                continue

            recall_items.append((
                raw_item_id,
                float(score)
            ))

            if len(recall_items) >= topk:
                break

        user_recall_items_dict[raw_uid] = recall_items

    # 与原文保持一致的保存文件名
    pickle.dump(
        user_recall_items_dict,
        open(
            save_path + "youtube_u2i_dict.pkl",
            "wb"
        )
    )

    # 记录实际拟合使用的用户数，供统一评估
    youtubednn_u2i_dict.train_users = int(
        np.unique(
            train_model_input["user_id"]
        ).size
    )

    return user_recall_items_dict

In [28]:
os.makedirs(save_path, exist_ok=True)

# 1. YouTubeDNN 直接 U2I 召回
user_multi_recall_dict["youtubednn_recall"] = (
    youtubednn_u2i_dict(
        all_click_df,
        query_clicks,
        topk=50
    )
)

# 2. 转成离线评估需要的长表
recall_dict = user_multi_recall_dict["youtubednn_recall"]

predictions = pd.DataFrame(
    [
        (user_id, item_id, score)
        for user_id, item_list in recall_dict.items()
        for item_id, score in item_list
    ],
    columns=[
        "user_id",
        "click_article_id",
        "pred_score"
    ]
)

# 3. 使用实际参与模型拟合的用户数
train_users = youtubednn_u2i_dict.train_users

metrics = evaluate_predictions(
    predictions,
    targets,
    ks=[5, 50],
    train_users=train_users
)

print(metrics)

# 4. 可选：使用已有的公共保存函数
save_predictions(
    predictions,
    save_path + "youtube_u2i_predictions.csv"
)

Build YoutubeDNN samples: 100%|██████████| 200000/200000 [00:08<00:00, 23106.99it/s]




8/8 [==============================] - 0s 15ms/step
User embedding: (200000, 16)
Item embedding: (29986, 16)


YoutubeDNN U2I recall: 100%|██████████| 50000/50000 [00:01<00:00, 45893.13it/s]


{'train_users': 179657, 'validation_users': 50000, 'R@5': 0.03736, 'R@50': 0.1862, 'MRR@5': 0.01749866666666667, 'MRR@50': 0.026704761148043637}


WindowsPath('results/youtube_u2i_predictions.csv')

### itemcf recall

下面使用协同过滤的思想，给用户召回与其历史文章相似的文章。 这里在召回的时候，也是用了关联规则的方式：

- 考虑相似文章与历史点击文章顺序的权重(细节看代码)
- 考虑文章创建时间的权重，也就是考虑相似文章与历史点击文章创建时间差的权重
- 考虑文章内容相似度权重(使用Embedding计算相似文章相似度，但是这里需要注意，在Embedding的时候并没有计算所有商品两两之间的相似度，所以相似的文章与历史点击文章不存在相似度，需要做特殊处理)

In [30]:
# ============================================================
# itemCF recall：定义通用的基于文章相似度的召回函数
# ============================================================

def item_based_recommend(
    user_id,
    user_item_time_dict,
    i2i_sim,
    sim_item_topk,
    recall_item_num,
    item_topk_click,
    item_created_time_dict,
    emb_i2i_sim
):
    """
    根据用户历史点击文章及 i2i 相似度生成召回结果。

    返回：
        [(article_id, score), ...]
        按 score 从高到低排序。
    """

    # 1. 获取用户历史交互文章
    user_hist_items = user_item_time_dict[user_id]
    user_hist_items_ = {
        item for item, _ in user_hist_items
    }

    item_rank = {}

    # 2. 遍历用户的历史点击
    for loc, (i, click_time) in enumerate(user_hist_items):

        # 取历史文章 i 最相似的 TopK 篇文章
        # 使用 get 避免个别 item 缺少相似度记录时 KeyError
        sim_items = sorted(
            i2i_sim.get(i, {}).items(),
            key=lambda x: x[1],
            reverse=True
        )[:sim_item_topk]

        for j, wij in sim_items:

            # 排除用户已经点击过的文章
            if j in user_hist_items_:
                continue

            # 3. 文章创建时间差权重（保留原文公式）
            created_time_weight = np.exp(
                0.8 ** np.abs(
                    item_created_time_dict[i]
                    - item_created_time_dict[j]
                )
            )

            # 4. 历史点击位置权重（保留原文公式）
            loc_weight = (
                0.9 ** (len(user_hist_items) - loc)
            )

            # 5. 内容 Embedding 相似度权重
            content_weight = 1.0

            if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                content_weight += emb_i2i_sim[i][j]

            if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                content_weight += emb_i2i_sim[j][i]

            # 6. 累加候选文章得分
            item_rank.setdefault(j, 0)

            item_rank[j] += (
                created_time_weight
                * loc_weight
                * content_weight
                * wij
            )

    # 7. 候选不足时，使用热门文章补全
    if len(item_rank) < recall_item_num:

        for loc, item in enumerate(item_topk_click):

            # 修复原文的 item in item_rank.items()
            # 同时避免重复推荐已点击文章
            if item in user_hist_items_ or item in item_rank:
                continue

            item_rank[item] = -loc - 100

            if len(item_rank) == recall_item_num:
                break

    # 8. 按得分降序返回最终候选
    item_rank = sorted(
        item_rank.items(),
        key=lambda x: x[1],
        reverse=True
    )[:recall_item_num]

    return item_rank

In [31]:
# ============================================================
# ItemCF sim 召回 + Embedding sim 召回
# ============================================================
os.makedirs(save_path, exist_ok=True)

# ------------------------------------------------------------
# 1. 获取固定验证用户
# ------------------------------------------------------------
# 不再执行 get_hist_and_last_click()
# query_clicks 已经排除了 targets 中的真实标签

query_users = query_clicks["user_id"].unique()

# 用户点击历史只需构造 5 万验证用户
# 与原来从全量 all_click_df 构造 20 万用户字典相比更省资源
user_item_time_dict = get_user_item_time(query_clicks)

# 热门文章必须基于整个 fit_clicks 统计
# all_click_df 就是 fit_clicks 的归一化副本
item_topk_click = get_item_topk_click(
    all_click_df,
    k=200
)

# ------------------------------------------------------------
# 2. 加载已经计算好的两种 i2i 相似度矩阵
# ------------------------------------------------------------
with open(
    os.path.join(save_path, "itemcf_i2i_sim.pkl"),
    "rb"
) as f:
    itemcf_i2i_sim = pickle.load(f)

with open(
    os.path.join(save_path, "emb_i2i_sim.pkl"),
    "rb"
) as f:
    emb_i2i_sim = pickle.load(f)

# ------------------------------------------------------------
# 3. 与原文保持一致的相似文章搜索参数
# ------------------------------------------------------------
sim_item_topk = 20

# 这里设为 50，便于统一离线 R@50
recall_item_num = 50

# ------------------------------------------------------------
# 4. 分别运行两条召回路线
# ------------------------------------------------------------
recall_routes = {
    "itemcf_sim_itemcf_recall": (
        itemcf_i2i_sim,
        "itemcf_recall_dict.pkl"
    ),
    "embedding_sim_item_recall": (
        emb_i2i_sim,
        "embedding_sim_item_recall.pkl"
    )
}

for route_name, (i2i_sim, filename) in recall_routes.items():

    user_recall_items_dict = {}

    for user in tqdm(query_users, desc=route_name):

        user_recall_items_dict[user] = item_based_recommend(
            user_id=user,
            user_item_time_dict=user_item_time_dict,
            i2i_sim=i2i_sim,
            sim_item_topk=sim_item_topk,
            recall_item_num=recall_item_num,
            item_topk_click=item_topk_click,
            item_created_time_dict=item_created_time_dict,
            emb_i2i_sim=emb_i2i_sim
        )

    # 保留原文的多路召回字典结构
    user_multi_recall_dict[route_name] = (
        user_recall_items_dict
    )

    # 保留原文的中间结果缓存文件名
    with open(
        os.path.join(save_path, filename),
        "wb"
    ) as f:
        pickle.dump(user_recall_items_dict, f)

    print(
        f"{route_name} 完成，"
        f"召回用户数：{len(user_recall_items_dict)}"
    )

itemcf_sim_itemcf_recall: 100%|██████████| 50000/50000 [04:43<00:00, 176.20it/s]


itemcf_sim_itemcf_recall 完成，召回用户数：50000


embedding_sim_item_recall: 100%|██████████| 50000/50000 [00:07<00:00, 6668.91it/s] 


embedding_sim_item_recall 完成，召回用户数：50000


In [32]:
# ============================================================
# 使用已有公共评估函数，比较两路召回效果
# ============================================================

from metric.evaluation import evaluate_predictions

# 相似度计算使用了完整 fit_clicks 中的用户点击
train_users = int(all_click_df["user_id"].nunique())

for route_name in recall_routes:

    recall_dict = user_multi_recall_dict[route_name]

    # 召回字典转为公共评估需要的长表
    predictions = pd.DataFrame(
        [
            (user_id, item_id, score)
            for user_id, items in recall_dict.items()
            for item_id, score in items
        ],
        columns=[
            "user_id",
            "click_article_id",
            "pred_score"
        ]
    )

    metrics = evaluate_predictions(
        predictions,
        targets,
        ks=[5, 50],
        train_users=train_users
    )

    print(f"\n===== {route_name} =====")
    print(f"R@5:   {metrics['R@5']:.6f}")
    print(f"R@50:  {metrics['R@50']:.6f}")
    print(f"MRR@5: {metrics['MRR@5']:.6f}")
    print(f"MRR@50:{metrics['MRR@50']:.6f}")


===== itemcf_sim_itemcf_recall =====
R@5:   0.291740
R@50:  0.614960
MRR@5: 0.169216
MRR@50:0.195650

===== embedding_sim_item_recall =====
R@5:   0.013060
R@50:  0.175340
MRR@5: 0.007759
MRR@50:0.015021


### usercf 召回

这里使用的关联规则主要是考虑相似用户的历史点击文章与被推荐用户历史点击商品的关系权重，而这里的关系就可以直接借鉴基于物品的协同过滤相似的做法，只不过这里是对被推荐物品关系的一个累加的过程，下面是使用的一些关系权重，及相关的代码：

- 计算被推荐用户历史点击文章与相似用户历史点击文章的相似度，文章创建时间差，相对位置的总和，作为各自的权重

In [33]:
# ============================================================
# UserCF recall：通用的 U2U2I 召回函数
# ============================================================

def user_based_recommend(
    user_id,
    user_item_time_dict,
    u2u_sim,
    sim_user_topk,
    recall_item_num,
    item_topk_click,
    item_created_time_dict,
    emb_i2i_sim
):
    """
    基于相似用户的历史点击召回文章。

    user_item_time_dict:
        {user_id: [(item_id, click_time), ...]}

    u2u_sim:
        {user_id: {similar_user_id: similarity, ...}}

    return:
        [(item_id, score), ...]
    """

    # 1. 获取目标用户的历史点击
    user_item_time_list = user_item_time_dict[user_id]

    user_hist_items = {
        item for item, _ in user_item_time_list
    }

    items_rank = {}

    # 2. 找最相似的 TopK 用户
    # 采样 UserCF 中，有些用户没有 U2U 结果
    similar_users = sorted(
        u2u_sim.get(user_id, {}).items(),
        key=lambda x: x[1],
        reverse=True
    )[:sim_user_topk]

    for sim_u, wuv in similar_users:

        # 采样用户之外的用户可能没有对应的历史
        if sim_u not in user_item_time_dict:
            continue

        # 3. 遍历相似用户的历史文章
        for i, click_time in user_item_time_dict[sim_u]:

            # 不推荐目标用户已点击的文章
            if i in user_hist_items:
                continue

            items_rank.setdefault(i, 0)

            loc_weight = 1.0
            content_weight = 1.0
            created_time_weight = 1.0

            # 4. 候选文章与目标用户历史进行关联加权
            for loc, (j, _) in enumerate(
                user_item_time_list
            ):

                # 历史点击位置权重：保留原文公式
                loc_weight += (
                    0.9 ** (len(user_item_time_list) - loc)
                )

                # 内容 Embedding 相似度权重
                if emb_i2i_sim.get(i, {}).get(j) is not None:
                    content_weight += emb_i2i_sim[i][j]

                if emb_i2i_sim.get(j, {}).get(i) is not None:
                    content_weight += emb_i2i_sim[j][i]

                # 文章创建时间权重：保留原文公式
                created_time_weight += np.exp(
                    0.8 * np.abs(
                        item_created_time_dict[i]
                        - item_created_time_dict[j]
                    )
                )

            # 5. 将来自不同相似用户的贡献累加
            items_rank[i] += (
                loc_weight
                * content_weight
                * created_time_weight
                * wuv
            )

    # 6. 热门文章补全
    if len(items_rank) < recall_item_num:

        for loc, item in enumerate(item_topk_click):

            # 修复原文判断，并排除已点击文章
            if item in items_rank or item in user_hist_items:
                continue

            items_rank[item] = -loc - 100

            if len(items_rank) >= recall_item_num:
                break

    # 7. 排序并返回 TopK
    items_rank = sorted(
        items_rank.items(),
        key=lambda x: x[1],
        reverse=True
    )[:recall_item_num]

    return items_rank

In [34]:
# ============================================================
# UserCF sim 召回
# ============================================================

os.makedirs(save_path, exist_ok=True)

# ------------------------------------------------------------
# 1. 召回用户：固定 5 万验证用户
# ------------------------------------------------------------
query_users = query_clicks["user_id"].unique()

# ------------------------------------------------------------
# 2. 构造全量 fit 用户的点击历史
# ------------------------------------------------------------
# 注意：与上一节 ItemCF 的使用方式不同
#
# UserCF 需要访问“相似用户”的历史文章。
# 因此这里必须包含全部 fit 用户，而不是只有 query_users。

user_item_time_dict = get_user_item_time(
    all_click_df
)

# ------------------------------------------------------------
# 3. 加载 UserCF 相似度及内容相似度
# ------------------------------------------------------------
with open(
    os.path.join(save_path, "usercf_u2u_sim.pkl"),
    "rb"
) as f:
    u2u_sim = pickle.load(f)

with open(
    os.path.join(save_path, "emb_i2i_sim.pkl"),
    "rb"
) as f:
    emb_i2i_sim = pickle.load(f)

# ------------------------------------------------------------
# 4. 召回参数
# ------------------------------------------------------------
sim_user_topk = 20
recall_item_num = 50

# 与前面的两路保持一致：
# 用全部 fit_clicks 统计热门文章
item_topk_click = get_item_topk_click(
    all_click_df,
    k=200
)

# ------------------------------------------------------------
# 5. 执行 UserCF 召回
# ------------------------------------------------------------
user_recall_items_dict = {}

for user in tqdm(
    query_users,
    desc="UserCF recall"
):
    user_recall_items_dict[user] = user_based_recommend(
        user_id=user,
        user_item_time_dict=user_item_time_dict,
        u2u_sim=u2u_sim,
        sim_user_topk=sim_user_topk,
        recall_item_num=recall_item_num,
        item_topk_click=item_topk_click,
        item_created_time_dict=item_created_time_dict,
        emb_i2i_sim=emb_i2i_sim
    )

# ------------------------------------------------------------
# 6. 保存中间结果
# ------------------------------------------------------------
with open(
    os.path.join(save_path, "usercf_u2u2i_recall.pkl"),
    "wb"
) as f:
    pickle.dump(user_recall_items_dict, f)

print(
    "UserCF 召回完成，用户数：",
    len(user_recall_items_dict)
)

UserCF recall: 100%|██████████| 50000/50000 [05:23<00:00, 154.45it/s]


UserCF 召回完成，用户数： 50000


In [35]:
# ============================================================
# UserCF 独立评估
# ============================================================

predictions = pd.DataFrame(
    [
        (int(user_id), int(item_id), float(score))
        for user_id, items in user_recall_items_dict.items()
        for item_id, score in items
    ],
    columns=[
        "user_id",
        "click_article_id",
        "pred_score"
    ]
)

metrics = evaluate_predictions(
    predictions,
    targets,
    ks=[5, 50],
    train_users=int(all_click_df["user_id"].nunique())
)

print("\n===== Sampled UserCF =====")
print(f"R@5:    {metrics['R@5']:.6f}")
print(f"R@50:   {metrics['R@50']:.6f}")
print(f"MRR@5:  {metrics['MRR@5']:.6f}")
print(f"MRR@50: {metrics['MRR@50']:.6f}")


===== Sampled UserCF =====
R@5:    0.120400
R@50:   0.369600
MRR@5:  0.066897
MRR@50: 0.083197


In [36]:
# ============================================================
# User Embedding Sim：Faiss 计算 U2U 相似度
# ============================================================

def u2u_embdding_sim(
    click_df,
    user_emb_dict,
    save_path,
    topk=21,
    query_users=None,
    batch_size=1024
):
    """
    使用 YouTubeDNN 学到的 User Embedding 计算 U2U 相似度。

    click_df:
        当前 fit_clicks 对应的点击数据。

    user_emb_dict:
        {原始 user_id: user_embedding}

    topk:
        Faiss 检索数量，包含用户自身。
        topk=21 通常得到 20 个其他相似用户。

    query_users:
        需要查询相似用户的集合。
        None 表示全部 fit 用户。

    batch_size:
        Faiss 每批查询的用户数。

    返回：
        {user_id: {similar_user_id: similarity, ...}}
    """

    if query_users is None:
        query_users = click_df["user_id"].unique()

    query_users = list(
        dict.fromkeys(int(u) for u in query_users)
    )

    # 1. 检查用户向量与当前 fit 数据是否一致
    fit_users = set(
        map(int, click_df["user_id"].unique())
    )

    emb_users = set(
        map(int, user_emb_dict)
    )

    if fit_users != emb_users:
        raise ValueError(
            "user_youtube_emb.pkl 与当前 fit_clicks "
            "的用户集合不一致，请检查训练数据或缓存。"
        )

    if not set(query_users).issubset(emb_users):
        raise ValueError(
            "某些 query 用户没有 User Embedding。"
        )

    if topk < 2:
        raise ValueError("topk 至少为 2（包含自身）。")

    # 2. 与原文一致：建立用户 ID 和 Faiss 索引的映射
    user_list = [
        int(u) for u in user_emb_dict
    ]

    user_index_2_rawid = dict(
        enumerate(user_list)
    )

    rawid_2_user_index = {
        u: i for i, u in enumerate(user_list)
    }

    user_emb_np = np.ascontiguousarray(
        np.asarray(
            list(user_emb_dict.values()),
            dtype=np.float32
        )
    )

    if (
        user_emb_np.ndim != 2
        or not np.isfinite(user_emb_np).all()
    ):
        raise ValueError("User Embedding 维度或数值异常。")

    # 3. L2 归一化，使用内积得到 cosine similarity
    user_emb_np /= np.maximum(
        np.linalg.norm(
            user_emb_np,
            axis=1,
            keepdims=True
        ),
        1e-12
    )

    # 4. 构建全量用户 Faiss 索引
    user_index = faiss.IndexFlatIP(
        user_emb_np.shape[1] # type: ignore
    )

    user_index.add(user_emb_np)

    search_k = min(topk, len(user_list))

    # 只存储 query 用户的 U2U 结果
    user_sim_dict = {
        u: {} for u in query_users
    }

    # 5. 分批检索验证用户的相似用户
    for start in tqdm(
        range(0, len(query_users), batch_size),
        desc="Faiss U2U"
    ):
        batch_users = query_users[
            start:start + batch_size
        ]

        batch_indices = [
            rawid_2_user_index[u]
            for u in batch_users
        ]

        batch_embs = np.ascontiguousarray(
            user_emb_np[batch_indices]
        )

        sim, idx = user_index.search(
            batch_embs,
            search_k
        )

        # 6. 恢复原始用户 ID
        for user_id, sim_values, user_indices in zip(
            batch_users, sim, idx
        ):
            for neighbor_idx, score in zip(
                user_indices, sim_values
            ):
                if neighbor_idx < 0:
                    continue

                neighbor_id = user_index_2_rawid[
                    int(neighbor_idx)
                ]

                # 按用户 ID 排除自己，不假设自己必在第一位
                if neighbor_id == user_id:
                    continue

                user_sim_dict[user_id][neighbor_id] = (
                    float(score)
                )

                if (
                    len(user_sim_dict[user_id])
                    >= topk - 1
                ):
                    break

    # 7. 保持原文的缓存文件名
    output_path = (
        Path(save_path) / "youtube_u2u_sim.pkl"
    )

    with output_path.open("wb") as f:
        pickle.dump(user_sim_dict, f)

    return user_sim_dict

In [37]:
# ============================================================
# 加载 DNN 用户向量，生成 U2U 相似度矩阵
# ============================================================

user_emb_path = (
    Path(save_path) / "user_youtube_emb.pkl"
)

if not user_emb_path.exists():
    raise FileNotFoundError(
        "找不到 user_youtube_emb.pkl。"
        "需要先完成YouTubeDNN 的训练和向量保存。"
    )

with user_emb_path.open("rb") as f:
    user_emb_dict = pickle.load(f)

query_users = query_clicks["user_id"].unique()

youtube_u2u_sim = u2u_embdding_sim(
    click_df=all_click_df,
    user_emb_dict=user_emb_dict,
    save_path=save_path,
    topk=21,
    query_users=query_users,
    batch_size=1024
)

print("用户向量总数:", len(user_emb_dict))
print("U2U 查询用户数:", len(youtube_u2u_sim))

print(
    "每位用户相似邻居数量:",
    pd.Series({
        u: len(neighbors)
        for u, neighbors in youtube_u2u_sim.items()
    }).describe()
)

Faiss U2U: 100%|██████████| 49/49 [00:04<00:00, 10.55it/s]


用户向量总数: 200000
U2U 查询用户数: 50000
每位用户相似邻居数量: count    50000.0
mean        20.0
std          0.0
min         20.0
25%         20.0
50%         20.0
75%         20.0
max         20.0
dtype: float64


In [38]:
# ============================================================
# YouTubeDNN UserCF：U2U2I 召回
# ============================================================

# 1. 全量 fit 用户历史
# 相似用户可能来自全部 20 万用户，不能仅使用 query_clicks
user_item_time_dict = get_user_item_time(
    all_click_df
)

# 2. 全量 fit 数据统计热门文章，用于候选补全
item_topk_click = get_item_topk_click(
    all_click_df,
    k=200
)

# 3. 复用前面计算的文章内容相似度矩阵
if "emb_i2i_sim" not in globals():
    with open(
        Path(save_path) / "emb_i2i_sim.pkl",
        "rb"
    ) as f:
        emb_i2i_sim = pickle.load(f)

# 4. 参数
sim_user_topk = 20
recall_item_num = 50

# 5. 为固定的 5 万验证用户召回
user_recall_items_dict = {}

for user in tqdm(
    query_users,
    desc="YouTubeDNN UserCF recall"
):
    user = int(user)

    user_recall_items_dict[user] = (
        user_based_recommend(
            user_id=user,
            user_item_time_dict=user_item_time_dict,
            u2u_sim=youtube_u2u_sim,
            sim_user_topk=sim_user_topk,
            recall_item_num=recall_item_num,
            item_topk_click=item_topk_click,
            item_created_time_dict=item_created_time_dict,
            emb_i2i_sim=emb_i2i_sim
        )
    )

# 6. 写入原来的多路召回字典
user_multi_recall_dict[
    "youtubednn_usercf_recall"
] = user_recall_items_dict

# 7. 保持原文的结果缓存名
with open(
    Path(save_path) / "youtubednn_usercf_recall.pkl",
    "wb"
) as f:
    pickle.dump(user_recall_items_dict, f)

print(
    "YouTubeDNN UserCF 召回用户数:",
    len(user_recall_items_dict)
)

print(
    "示例召回:",
    next(iter(user_recall_items_dict.items()))
)

YouTubeDNN UserCF recall: 100%|██████████| 50000/50000 [00:28<00:00, 1724.15it/s]


YouTubeDNN UserCF 召回用户数: 50000
示例召回: (0, [(208562, np.float64(25.687952282876815)), (348112, np.float64(7.37339891459046)), (233688, np.float64(7.372768141048944)), (336245, np.float64(7.346004993272656)), (348109, np.float64(3.695327126985216)), (128780, np.float64(3.6945559500583)), (202637, np.float64(3.6943786890568475)), (129237, np.float64(3.6938826161879956)), (234006, np.float64(3.6801485821299016)), (236512, np.float64(3.679959533948814)), (50405, np.float64(3.6780377054595568)), (320901, np.float64(3.677530621341699)), (297966, np.float64(3.6736438564860014)), (20691, np.float64(3.67272193206388)), (213897, np.float64(3.6726841510025077)), (202416, np.float64(3.672682147372254)), (157043, np.float64(3.6726317484587425)), (271262, np.float64(3.672465394729561)), (331116, np.float64(3.6717004701687994)), (202355, np.float64(3.671651927052642)), (234698, -100), (123909, -101), (336221, -102), (96210, -103), (336223, -104), (183176, -105), (168623, -106), (124749, -109), (235616,

In [39]:
# ============================================================
# YouTubeDNN UserCF 离线评估
# ============================================================

recall_dict = user_multi_recall_dict[
    "youtubednn_usercf_recall"
]

predictions = pd.DataFrame(
    [
        (user_id, item_id, score)
        for user_id, items in recall_dict.items()
        for item_id, score in items
    ],
    columns=[
        "user_id",
        "click_article_id",
        "pred_score"
    ]
)

# 根据我们此前的 DNN 滑窗训练逻辑：
# 只有至少存在两条可见点击的用户才能构成监督训练样本
train_users = int(
    (all_click_df.groupby("user_id").size() >= 2).sum()
)

metrics = evaluate_predictions(
    predictions,
    targets,
    ks=[5, 50],
    train_users=train_users
)

print("\n===== YouTubeDNN UserCF =====")
print(metrics)


===== YouTubeDNN UserCF =====
{'train_users': 179657, 'validation_users': 50000, 'R@5': 0.06104, 'R@50': 0.24132, 'MRR@5': 0.03271, 'MRR@50': 0.04433735389468254}


## 冷启动

冷启动问题可以分成三类：文章冷启动，用户冷启动，系统冷启动。

- 文章冷启动：对于一个平台系统新加入的文章，该文章没有任何的交互记录，如何推荐给用户的问题。(对于我们场景可以认为是，日志数据中没有出现过的文章都可以认为是冷启动的文章)

- 用户冷启动：对于一个平台系统新来的用户，该用户还没有文章的交互信息，如何给该用户进行推荐。(对于我们场景就是，测试集中的用户是否在测试集对应的log数据中出现过，如果没有出现过，那么可以认为该用户是冷启动用户。但是有时候并没有这么严格，我们也可以自己设定某些指标来判别哪些用户是冷启动用户，比如通过使用时长，点击率，留存率等等)

- 系统冷启动：就是对于一个平台刚上线，还没有任何的相关历史数据，此时就是系统冷启动，其实也就是前面两种的一个综合。
 
**当前场景下冷启动问题的分析：**

对当前的数据进行分析会发现，日志中所有出现过的点击文章只有3w多个，而整个文章库中却有30多万，那么测试集中的用户最后一次点击是否会点击没有出现在日志中的文章呢？如果存在这种情况，说明用户点击的文章之前没有任何的交互信息，这也就是我们所说的文章冷启动。通过数据分析还可以发现，测试集用户只有一次点击的数据占得比例还不少，其实仅仅通过用户的一次点击就给用户推荐文章使用模型的方式也是比较难的，这里其实也可以考虑用户冷启动的问题，但是这里只给出物品冷启动的一些解决方案及代码，关于用户冷启动的话提一些可行性的做法。

1. 文章冷启动(没有冷启动的探索问题)

其实我们这里不是为了做文章的冷启动而做冷启动，而是猜测用户可能会点击一些没有在log数据中出现的文章，我们要做的就是如何从将近27万的文章中选择一些文章作为用户冷启动的文章，这里其实也可以看成是一种召回策略，我们这里就采用简单的比较好理解的基于规则的召回策略来获取用户可能点击的未出现在log数据中的文章。 现在的问题变成了：如何给每个用户考虑从27万个商品中获取一小部分商品？随机选一些可能是一种方案。下面给出一些参考的方案。

- 首先基于Embedding召回一部分与用户历史相似的文章
- 从基于Embedding召回的文章中通过一些规则过滤掉一些文章，使得留下的文章用户更可能点击。我们这里的规则，可以是，留下那些与用户历史点击文章主题相同的文章，或者字数相差不大的文章。并且留下的文章尽量是与测试集用户最后一次点击时间更接近的文章，或者是当天的文章也行。
  
2. 用户冷启动

这里对测试集中的用户点击数据进行分析会发现，测试集中有百分之20的用户只有一次点击，那么这些点击特别少的用户的召回是不是可以单独做一些策略上的补充呢？或者是在排序后直接基于规则加上一些文章呢？这些都可以去尝试，这里没有提供具体的做法。

**注意：**

这里看似和基于embedding计算的item之间相似度然后做itemcf是一致的，但是现在我们的目的不一样，我们这里的目的是找到相似的向量，并且还没有出现在log日志中的商品，再加上一些其他的冷启动的策略，这里需要找回的数量会偏多一点，不然被筛选完之后可能都没有文章了

In [40]:
# ============================================================
# 冷启动阶段 1：通过内容 Embedding 扩大候选池
# ============================================================

from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

save_dir = Path(save_path)
save_dir.mkdir(parents=True, exist_ok=True)

# 1. 固定离线验证用户
query_users = query_clicks["user_id"].unique()

# 2. 用户历史只使用可见点击，不能包含 targets
user_item_time_dict = get_user_item_time(query_clicks)

# 3. 加载原文已经计算好的 Embedding I2I 相似度
with open(save_dir / "emb_i2i_sim.pkl", "rb") as f:
    emb_i2i_sim = pickle.load(f)

# 4. 冷启动原始候选参数，保留原文设置
sim_item_topk = 150
raw_recall_num = 100

# 热门补全来自完整 fit_clicks
item_topk_click = get_item_topk_click(
    all_click_df,
    k=50
)

# 5. 先使用 Embedding 召回较多文章
cold_raw_recall_dict = {}

for user in tqdm(
    query_users,
    desc="Cold-start raw recall"
):
    user = int(user)

    cold_raw_recall_dict[user] = item_based_recommend(
        user_id=user,
        user_item_time_dict=user_item_time_dict,
        i2i_sim=emb_i2i_sim,
        sim_item_topk=sim_item_topk,
        recall_item_num=raw_recall_num,
        item_topk_click=item_topk_click,
        item_created_time_dict=item_created_time_dict,
        emb_i2i_sim=emb_i2i_sim
    )

# 6. 保存原始候选，保持原文文件名
with open(
    save_dir / "cold_start_items_raw_dict.pkl",
    "wb"
) as f:
    pickle.dump(cold_raw_recall_dict, f)

print("原始召回用户数:", len(cold_raw_recall_dict))
print(
    "原始候选总数:",
    sum(len(items) for items in cold_raw_recall_dict.values())
)

Cold-start raw recall: 100%|██████████| 50000/50000 [00:06<00:00, 8009.04it/s] 


原始召回用户数: 50000
原始候选总数: 3769705


In [41]:
# ============================================================
# 冷启动阶段 2：定义基于规则的过滤函数
# ============================================================

def get_click_article_ids_set(all_click_df):
    return set(all_click_df["click_article_id"].values)


def cold_start_items(
    user_recall_items_dict,
    user_hist_item_typs_dict,
    user_hist_item_words_dict,
    user_last_item_created_time_dict,
    item_type_dict,
    item_words_dict,
    item_created_time_dict,
    click_article_ids_set,
    recall_item_num
):
    """
    原文冷启动过滤逻辑。

    注意：
    这里传入的 item_created_time_dict 必须是
    原始毫秒时间戳字典，而不是之前用于 ItemCF
    评分的归一化时间字典。
    """

    cold_start_user_items_dict = {}

    # 90 天对应的毫秒数
    max_time_gap_ms = 90 * 24 * 60 * 60 * 1000

    for user, item_list in tqdm(
        user_recall_items_dict.items(),
        desc="Cold-start filtering"
    ):
        hist_item_type_set = (
            user_hist_item_typs_dict.get(user, set())
        )

        hist_mean_words = (
            user_hist_item_words_dict.get(user)
        )

        hist_last_item_created_time = (
            user_last_item_created_time_dict.get(user)
        )

        filtered_items = []

        for item, score in item_list:

            # 规则 1：不能出现在 fit_clicks 中
            if item in click_article_ids_set:
                continue

            curr_item_type = item_type_dict.get(item)
            curr_item_words = item_words_dict.get(item)
            curr_item_created_time = (
                item_created_time_dict.get(item)
            )

            # 属性缺失时无法执行原文规则
            if (
                curr_item_type not in hist_item_type_set
                or pd.isna(hist_mean_words)
                or pd.isna(curr_item_words)
                or pd.isna(hist_last_item_created_time)
                or pd.isna(curr_item_created_time)
            ):
                continue

            # 规则 2：历史兴趣类别匹配
            # 已由上面的 curr_item_type 检查完成

            # 规则 3：字数与历史平均值相差不超过 200
            if abs(
                curr_item_words - hist_mean_words
            ) > 200:
                continue

            # 规则 4：创建时间与最后阅读文章
            # 的创建时间相差不超过 90 天
            if abs(
                curr_item_created_time
                - hist_last_item_created_time
            ) > max_time_gap_ms:
                continue

            filtered_items.append((item, score))

        # 保持原有召回评分，只做排序和截断
        cold_start_user_items_dict[user] = sorted(
            filtered_items,
            key=lambda x: x[1],
            reverse=True
        )[:recall_item_num]

    # 保持原文的缓存文件名
    with open(
        Path(save_path) / "cold_start_user_items_dict.pkl",
        "wb"
    ) as f:
        pickle.dump(cold_start_user_items_dict, f)

    return cold_start_user_items_dict

In [42]:
# ============================================================
# 冷启动阶段 3：准备属性、执行过滤
# ============================================================

# 1. 重新读取原始文章属性
# 此前 item_info_df 可能已被 get_item_info_dict() 原地归一化
raw_item_info_df = get_item_info_df(data_path)

# 2. 文章属性字典
# item_type_dict / item_words_dict 之前已有，
# 这里直接继续复用。

# 单独建立原始毫秒时间戳字典
raw_item_created_time_dict = dict(zip(
    raw_item_info_df["click_article_id"],
    raw_item_info_df["created_at_ts"]
))

# 3. 根据 query_clicks 构造用户历史文章属性
query_click_info_df = query_clicks.merge(
    raw_item_info_df[
        ["click_article_id", "category_id", "words_count"]
    ],
    how="left",
    on="click_article_id"
)

# 用户点击过的文章类别集合
user_hist_item_typs_dict = (
    query_click_info_df
    .groupby("user_id")["category_id"]
    .agg(set)
    .to_dict()
)

# 用户历史阅读文章的平均字数
user_hist_item_words_dict = (
    query_click_info_df
    .groupby("user_id")["words_count"]
    .mean()
    .to_dict()
)

# 4. 找出每位用户最后一条可见点击
# 必须使用 query_clicks，而不是 targets
user_last_click_df = (
    query_clicks
    .sort_values(
        ["user_id", "click_timestamp"],
        kind="mergesort"
    )
    .groupby("user_id")
    .tail(1)
)

# 获取用户最后点击文章的原始创建时间
user_last_item_created_time_dict = dict(zip(
    user_last_click_df["user_id"],
    user_last_click_df["click_article_id"].map(
        raw_item_created_time_dict
    )
))

# 5. fit_clicks 中出现过的所有文章
# 这里用全量 fit，而不只是 query_clicks
click_article_ids_set = get_click_article_ids_set(
    all_click_df
)

# 6. 运行冷启动规则过滤
cold_recall_num = 100

cold_start_user_items_dict = cold_start_items(
    user_recall_items_dict=cold_raw_recall_dict,
    user_hist_item_typs_dict=user_hist_item_typs_dict,
    user_hist_item_words_dict=user_hist_item_words_dict,
    user_last_item_created_time_dict=(
        user_last_item_created_time_dict
    ),
    item_type_dict=item_type_dict,
    item_words_dict=item_words_dict,
    item_created_time_dict=raw_item_created_time_dict,
    click_article_ids_set=click_article_ids_set,
    recall_item_num=cold_recall_num
)

# 7. 加入原文多路召回字典
user_multi_recall_dict[
    "cold_start_recall"
] = cold_start_user_items_dict

# 8. 检查结果
users_with_cold_items = sum(
    len(items) > 0
    for items in cold_start_user_items_dict.values()
)

total_cold_items = sum(
    len(items)
    for items in cold_start_user_items_dict.values()
)

print("验证用户总数:", len(query_users))
print("有冷启动候选的用户数:", users_with_cold_items)
print("冷启动候选总数:", total_cold_items)

print(
    "有候选用户占比:",
    f"{users_with_cold_items / len(query_users):.2%}"
)

# 验证最终候选均未出现在 fit_clicks 中
assert all(
    item not in click_article_ids_set
    for items in cold_start_user_items_dict.values()
    for item, score in items
)

print("冷启动候选过滤检查通过")

Cold-start filtering: 100%|██████████| 50000/50000 [00:02<00:00, 22047.74it/s]


验证用户总数: 50000
有冷启动候选的用户数: 45076
冷启动候选总数: 457015
有候选用户占比: 90.15%
冷启动候选过滤检查通过


In [43]:
# ============================================================
# 冷启动召回简单离线评估
# ============================================================

recall_dict = user_multi_recall_dict[
    "cold_start_recall"
]

predictions = pd.DataFrame(
    [
        (user_id, item_id, score)
        for user_id, items in recall_dict.items()
        for item_id, score in items
    ],
    columns=[
        "user_id",
        "click_article_id",
        "pred_score"
    ]
)

train_users = int(
    all_click_df["user_id"].nunique()
)

# 1. 全体验证用户的召回效果
overall_metrics = evaluate_predictions(
    predictions,
    targets,
    ks=[5, 50],
    train_users=train_users
)

print("===== Cold-start Overall =====")
print(overall_metrics)

# 2. 只考察 fit_clicks 从未出现的目标文章
cold_targets = targets[
    ~targets["click_article_id"].isin(
        click_article_ids_set
    )
].copy()

if not cold_targets.empty:
    cold_metrics = evaluate_predictions(
        predictions,
        cold_targets,
        ks=[5, 50],
        train_users=train_users
    )

    print("\n===== Cold-item Targets Only =====")
    print("冷启动验证目标数:", len(cold_targets))
    print(cold_metrics)

===== Cold-start Overall =====
{'train_users': 200000, 'validation_users': 50000, 'R@5': 0.00108, 'R@50': 0.00112, 'MRR@5': 0.0007936666666666667, 'MRR@50': 0.0007981904761904761}

===== Cold-item Targets Only =====
冷启动验证目标数: 1153
{'train_users': 200000, 'validation_users': 1153, 'R@5': 0.046834345186470075, 'R@50': 0.04856895056374675, 'MRR@5': 0.034417461694131256, 'MRR@50': 0.03461363730227563}


## 多路召回合并

多路召回合并就是将前面所有的召回策略得到的用户文章列表合并起来，下面是对前面所有召回结果的汇总

- 基于itemcf计算的item之间的相似度sim进行的召回

- 基于embedding搜索得到的item之间的相似度进行的召回

- YoutubeDNN召回

- YoutubeDNN得到的user之间的相似度进行的召回

- 基于冷启动策略的召回

**注意：**
在做召回评估的时候就会发现有些召回的效果不错有些召回的效果很差，所以对每一路召回的结果，我们可以认为的定义一些权重，来做最终的相似度融合

In [44]:
# ============================================================
# 多路召回融合
# ============================================================

def combine_recall_results(
    user_multi_recall_dict,
    weight_dict=None,
    topk=25,
    query_users=None
):
    """
    五路召回结果融合。

    user_multi_recall_dict:
        {
            method: {
                user_id: [(article_id, score), ...]
            }
        }

    weight_dict:
        各路召回权重；None 表示所有路线权重均为 1。

    topk:
        最终每个用户保留的候选文章数。

    query_users:
        当前需要预测的用户。
        None 时使用各路召回用户的并集。

    返回：
        {user_id: [(article_id, fused_score), ...]}
    """

    # 1. 每路召回结果按用户分别归一化
    def norm_user_recall_items_sim(sorted_item_list):

        if not sorted_item_list:
            return []

        scores = [
            float(score)
            for _, score in sorted_item_list
        ]

        if not all(math.isfinite(score) for score in scores):
            raise ValueError("召回结果存在 NaN 或 Inf 分数")

        min_sim = min(scores)
        max_sim = max(scores)

        # 单候选或所有候选同分时，统一赋值为 1
        if max_sim == min_sim:
            return [
                (int(item), 1.0)
                for item, _ in sorted_item_list
            ]

        # Min-Max 归一化
        return [
            (
                int(item),
                (float(score) - min_sim)
                / (max_sim - min_sim)
            )
            for item, score in sorted_item_list
        ]

    # 2. 默认采用等权融合
    if weight_dict is None:
        weight_dict = {
            method: 1.0
            for method in user_multi_recall_dict
        }

    if topk < 1:
        raise ValueError("topk 必须大于 0")

    missing_methods = (
        set(weight_dict) - set(user_multi_recall_dict)
    )

    if missing_methods:
        raise ValueError(
            f"缺少召回路线：{sorted(missing_methods)}"
        )

    if any(
        not math.isfinite(float(weight))
        or float(weight) < 0
        for weight in weight_dict.values()
    ):
        raise ValueError("融合权重必须是有限的非负数")

    # 3. 确定需要融合的用户
    if query_users is None:
        users = sorted({
            int(user)
            for recall_dict in user_multi_recall_dict.values()
            for user in recall_dict
        })
    else:
        users = list(dict.fromkeys(
            int(user) for user in query_users
        ))

    # 4. 按用户逐个完成五路融合，避免保存额外的全量中间字典
    final_recall_items_dict_rank = {}

    print("多路召回合并...")

    for user in tqdm(users, desc="Fusion Recall"):

        item_rank = {}

        for method, recall_method_weight in weight_dict.items():

            if recall_method_weight == 0:
                continue

            sorted_item_list = (
                user_multi_recall_dict[method].get(
                    user, []
                )
            )

            # 每路、每个用户独立归一化
            norm_item_list = norm_user_recall_items_sim(
                sorted_item_list
            )

            # 同一篇文章跨路得分累加
            for item, score in norm_item_list:

                item_rank[item] = (
                    item_rank.get(item, 0.0)
                    + recall_method_weight * score
                )

        # 5. 按融合得分降序，取 TopK
        # 同分时按文章 ID 升序，保证结果可复现
        final_recall_items_dict_rank[user] = sorted(
            item_rank.items(),
            key=lambda x: (-x[1], x[0])
        )[:topk]

    # 6. 保存最终排序、截断后的召回结果
    output_dir = Path(save_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    with open(
        output_dir / "final_recall_items_dict.pkl",
        "wb"
    ) as f:
        pickle.dump(
            final_recall_items_dict_rank,
            f
        )

    return final_recall_items_dict_rank

In [45]:
# ============================================================
# 五路融合调用
# ============================================================

weight_dict = {
    "itemcf_sim_itemcf_recall": 1.0,
    "embedding_sim_item_recall": 1.0,
    "youtubednn_recall": 1.0,
    "youtubednn_usercf_recall": 1.0,
    "cold_start_recall": 1.0
}

query_users = query_clicks["user_id"].unique()

# 检查所有路线是否完成
missing_routes = [
    method
    for method in weight_dict
    if (
        method not in user_multi_recall_dict
        or not user_multi_recall_dict[method]
    )
]

if missing_routes:
    raise ValueError(
        f"以下召回路线尚未生成结果：{missing_routes}"
    )

# 检查是否覆盖全部验证用户
query_user_set = set(map(int, query_users))

for method in weight_dict:
    route_user_set = set(
        map(int, user_multi_recall_dict[method])
    )

    missing_users = query_user_set - route_user_set

    if missing_users:
        raise ValueError(
            f"{method} 缺少 {len(missing_users)} 个验证用户"
        )

    users_with_items = sum(
        len(items) > 0
        for items in user_multi_recall_dict[method].values()
    )

    print(
        f"{method}: "
        f"{users_with_items} 个用户有召回候选"
    )

# 原文采用等权融合，保留 150 篇候选
final_recall_items_dict_rank = combine_recall_results(
    user_multi_recall_dict,
    weight_dict=weight_dict,
    topk=150,
    query_users=query_users
)

print(
    "融合用户数:",
    len(final_recall_items_dict_rank)
)

print(
    "首位用户召回结果:",
    next(iter(final_recall_items_dict_rank.items()))[1][:5]
)

itemcf_sim_itemcf_recall: 50000 个用户有召回候选
embedding_sim_item_recall: 50000 个用户有召回候选
youtubednn_recall: 50000 个用户有召回候选
youtubednn_usercf_recall: 50000 个用户有召回候选
cold_start_recall: 45076 个用户有召回候选
多路召回合并...


Fusion Recall: 100%|██████████| 50000/50000 [00:06<00:00, 7749.69it/s]


融合用户数: 50000
首位用户召回结果: [(30955, 1.9992333049973956), (31966, 1.9848234180979034), (336245, 1.2660689155653824), (331116, 1.2577316855176057), (233688, 1.135475766264307)]


In [46]:
# ============================================================
# 多路融合离线评估
# ============================================================

from metric.evaluation import evaluate_predictions

# 仅取融合结果的 Top50 参与当前公共评估
predictions = pd.DataFrame(
    [
        (user_id, item_id, rank)
        for user_id, items in final_recall_items_dict_rank.items()
        for rank, (item_id, score) in enumerate(
            items[:50], start=1
        )
    ],
    columns=[
        "user_id",
        "click_article_id",
        "recall_rank"
    ]
)

# 全部 fit 用户均参与了 ItemCF 等行为统计
train_users = int(fit_clicks["user_id"].nunique())

fusion_metrics = evaluate_predictions(
    predictions,
    targets,
    ks=[5, 50],
    train_users=train_users
)

print("\n===== Multi-Recall Fusion =====")

for metric, value in fusion_metrics.items():
    if metric in ("train_users", "validation_users"):
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.6f}")


===== Multi-Recall Fusion =====
train_users: 200000
validation_users: 50000
R@5: 0.131800
R@50: 0.477960
MRR@5: 0.072010
MRR@50: 0.093873
